
# Component 3 (continued): Characterizing Wind/Solar Uncertainty

`forecasting_theory.ipynb` covers the deterministic point forecasters and checks the
probabilistic (quantile) LSTM's calibration for **load only**. This notebook picks up two
things that notebook deliberately left out of scope:

1. Quantifying, with real numbers rather than just visual impression, exactly how much
   more "stochastic" wind and solar are than load -- the whole premise for why a
   probabilistic forecast matters more for renewables than for load.
2. Extending the calibration check across **all three targets** now that
   `probabilistic.py --target wind_mw` and `--target solar_mw` have real trained
   checkpoints (they didn't yet when `forecasting_theory.ipynb` was first built), plus a
   look at how the [q10, q90] band's width itself behaves -- e.g. whether solar's
   uncertainty correctly shrinks toward 0 at night.

It closes with a simple, honestly-scoped Monte Carlo scenario generator built from
historical residuals -- useful for a "what might tomorrow's wind/solar profile look like"
grid-planning exercise, but explicitly NOT a physical weather model, and the notebook says
so rather than implying more than it is.


In [ ]:
import os, sys

def find_project_root(start):
    '''Walk up from `start` until a directory containing both 'src' and 'data' is
    found. Safe to re-run: once cwd IS the project root, it's found immediately.'''
    path = os.path.abspath(start)
    for _ in range(4):
        if os.path.isdir(os.path.join(path, "src")) and os.path.isdir(os.path.join(path, "data")):
            return path
        path = os.path.dirname(path)
    raise RuntimeError(
        "Could not locate the project root (a folder containing both 'src' and 'data') "
        "within 4 levels above the current directory. Run this notebook from inside "
        "smart-grid-rl/notebooks/, or adjust find_project_root's search depth."
    )

PROJECT_ROOT = find_project_root(os.getcwd())
os.chdir(PROJECT_ROOT)
if os.path.join(PROJECT_ROOT, "src") not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
print("Project root:", PROJECT_ROOT)

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from forecasting.lstm_model import SequenceDataset, Standardizer, TIME_FEATURE_COLS, NON_NEGATIVE_TARGETS
from forecasting.probabilistic import QuantileLSTM

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

merged = pd.read_csv(os.path.join("data", "processed", "grid_merged.csv"), parse_dates=["timestamp"])
print(f"{len(merged):,} rows, {merged['timestamp'].min()} -> {merged['timestamp'].max()}")


## 1. How much more "stochastic" are wind and solar than load, really?

Two simple, real diagnostics, computed directly on the raw series (no model involved
yet):

- **Lag-1 autocorrelation**: how well does the value 15 minutes ago predict the value now?
  A value near 1.0 means the series is smooth/persistent step-to-step; a lower value means
  it jumps around more from one reading to the next.
- **Noise ratio**: the standard deviation of *first differences* (`x[t] - x[t-1]`) divided
  by the standard deviation of the series itself. A series that's mostly slow drift has a
  low ratio; a series that jitters a lot relative to its own overall spread has a high one.

Both are computed on daytime-only solar readings too (`solar_mw > 0`), since otherwise the
long run of exact zeros at night would understate how volatile solar actually is once the
sun is up -- that flat run is a diurnal pattern, not evidence of smoothness.


In [ ]:
def stochasticity_stats(series: pd.Series) -> dict:
    diffs = series.diff().dropna()
    return {
        "lag1_autocorr": series.autocorr(lag=1),
        "noise_ratio": diffs.std() / series.std(),
    }

rows = {
    "load_mw": stochasticity_stats(merged["load_mw"]),
    "wind_mw": stochasticity_stats(merged["wind_mw"]),
    "solar_mw (all hours)": stochasticity_stats(merged["solar_mw"]),
    "solar_mw (daytime only)": stochasticity_stats(merged.loc[merged["solar_mw"] > 0, "solar_mw"]),
}
pd.DataFrame(rows).T


A lower lag-1 autocorrelation and a higher noise ratio for wind (and daytime solar) than
for load is exactly the quantitative version of "wind/solar are harder to forecast" --
and it's a property of the raw physical process itself, visible before any model is
involved, not an artifact of how well or badly a particular forecaster performs.



## 2. Probabilistic forecast calibration, all three targets

Same evaluation `probabilistic.py`'s own `__main__` block does for whichever single
target you pass it with `--target`, run here for all three side by side so the
differences are directly comparable in one table. Requires
`outputs/probabilistic_lstm_{target}_forecaster.pt` for each target to already exist
(`python src/forecasting/probabilistic.py --target <target>`).


In [ ]:
def load_quantile_checkpoint(target):
    short = target.replace("_mw", "")
    path = os.path.join("outputs", f"probabilistic_lstm_{short}_forecaster.pt")
    ckpt = torch.load(path, map_location="cpu")
    scaler = Standardizer()
    scaler.mean, scaler.std = ckpt["scaler_mean"], ckpt["scaler_std"]
    model = QuantileLSTM(input_size=1 + len(TIME_FEATURE_COLS), quantiles=ckpt["quantiles"])
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model, scaler, ckpt["seq_len"], ckpt["horizon"]


def evaluate_target(target):
    val_path = os.path.join("data", "processed", f"{target}_val.csv")
    val_raw = pd.read_csv(val_path, parse_dates=["timestamp"])[["timestamp", target]]

    model, scaler, seq_len, horizon = load_quantile_checkpoint(target)
    ds = SequenceDataset(val_raw, target_col=target, standardizer=scaler, seq_len=seq_len, horizon=horizon)

    preds_mw, actuals_mw, timestamps = [], [], []
    with torch.no_grad():
        for idx in range(len(ds)):
            window, y = ds[idx]
            pred_scaled = model(window.unsqueeze(0)).numpy()[0]
            preds_mw.append(scaler.inverse_transform(pred_scaled))
            actuals_mw.append(scaler.inverse_transform(np.array([y.item()]))[0])
            timestamps.append(ds.target_timestamp(idx))
    preds_mw = np.array(preds_mw)
    actuals_mw = np.array(actuals_mw)

    n_crossed = int(np.sum((preds_mw[:, 0] > preds_mw[:, 1]) | (preds_mw[:, 1] > preds_mw[:, 2])))
    sorted_preds = np.sort(preds_mw, axis=1)

    n_clipped = 0
    if target in NON_NEGATIVE_TARGETS:
        n_clipped = int((sorted_preds < 0).sum())
        sorted_preds = np.clip(sorted_preds, 0.0, None)

    q10, q50, q90 = sorted_preds[:, 0], sorted_preds[:, 1], sorted_preds[:, 2]
    coverage = float(np.mean((actuals_mw >= q10) & (actuals_mw <= q90)))
    below_median = float(np.mean(actuals_mw <= q50))

    return {
        "target": target,
        "n_predictions": len(q10),
        "crossing_pct": 100 * n_crossed / len(q10),
        "n_clipped": n_clipped,
        "coverage_pct": 100 * coverage,
        "pct_below_median": 100 * below_median,
        "mean_interval_width_mw": float(np.mean(q90 - q10)),
        "median_mae_mw": float(np.mean(np.abs(q50 - actuals_mw))),
    }, timestamps, q10, q50, q90, actuals_mw


results = {}
per_target_arrays = {}
for target in ["load_mw", "wind_mw", "solar_mw"]:
    try:
        summary, timestamps, q10, q50, q90, actuals_mw = evaluate_target(target)
        results[target] = summary
        per_target_arrays[target] = dict(timestamps=timestamps, q10=q10, q50=q50, q90=q90, actuals_mw=actuals_mw)
    except FileNotFoundError as e:
        print(f"Skipping {target}: {e}")

pd.DataFrame(results).T


**Reading `pct_below_median`**: a well-calibrated median should have close to 50% of
actual outcomes fall at or below it, by definition of what a median forecast means --
this is a second, independent calibration check beyond the 80%-coverage number, and the
two don't have to agree (a model can have the right *interval width* on average while its
*median* is still biased high or low).

**Reading `coverage_pct`**: 80% is the target, since [q10, q90] is supposed to bracket the
middle 80% of outcomes by construction (same reading as in `forecasting_theory.ipynb`).
Whichever target comes out furthest from 80% here is the one whose uncertainty band is
least trustworthy as-is -- worth stating plainly rather than glossing over, exactly as
`forecasting_theory.ipynb` already does for load alone.



## 3. Does the uncertainty band actually know it's nighttime?

A probabilistic solar forecaster that's actually learned the physical day/night cycle
should produce a near-zero-width [q10, q90] band at night (there's essentially no
uncertainty about whether the sun is up) and a much wider one at midday. This plots mean
interval width by hour of day for whichever targets were evaluated above -- the shape of
solar's curve here is the real test of whether the model learned that constraint, not
just clipped its way to non-negative outputs after the fact.


In [ ]:
fig, axes = plt.subplots(1, len(per_target_arrays), figsize=(5 * len(per_target_arrays), 4), squeeze=False)
axes = axes[0]

for ax, (target, arrs) in zip(axes, per_target_arrays.items()):
    hours = pd.Series([ts.hour + ts.minute / 60.0 for ts in arrs["timestamps"]])
    width = pd.Series(arrs["q90"] - arrs["q10"])
    by_hour = width.groupby(hours).mean()
    ax.plot(by_hour.index, by_hour.values, marker="o", markersize=3)
    ax.set_title(target)
    ax.set_xlabel("hour of day")
    ax.set_ylabel("mean [q10, q90] width (MW)")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 4. A simple Monte Carlo scenario generator (residual block bootstrap)

This is deliberately the simplest honest thing that could work, not a claim to weather
modeling: fit a smooth per-hour-of-day mean profile to the historical series (the same
diurnal average computed in `eda_grid_data.ipynb`), compute each historical day's
deviation from that profile as a *residual*, then build synthetic future days by
splicing together randomly-chosen historical residual days on top of the mean profile.
This preserves each hour's realistic spread and the rough shape of a day's worth of
correlated deviations (since whole days are resampled, not independent 15-min points),
without pretending to simulate actual weather physics.


In [ ]:
def build_scenarios(series_df: pd.DataFrame, target: str, n_scenarios: int = 20, seed: int = 0):
    df = series_df.copy()
    df["hour"] = df["timestamp"].dt.hour + df["timestamp"].dt.minute / 60.0
    df["date"] = df["timestamp"].dt.date

    hourly_mean = df.groupby("hour")[target].mean()
    df["profile"] = df["hour"].map(hourly_mean)
    df["residual"] = df[target] - df["profile"]

    # Only keep full, complete days (96 15-min steps) as resampling units, so a spliced-in
    # residual day always has a value for every hour of the synthetic day it's applied to.
    counts = df.groupby("date").size()
    full_dates = counts[counts == 96].index.tolist()
    if not full_dates:
        raise ValueError(f"No complete 96-step days available for {target} to resample from.")

    rng = np.random.default_rng(seed)
    hours_grid = np.sort(df["hour"].unique())
    profile_by_hour = hourly_mean.reindex(hours_grid).values

    scenarios = []
    for _ in range(n_scenarios):
        sampled_date = rng.choice(full_dates)
        day_residuals = df.loc[df["date"] == sampled_date].set_index("hour")["residual"].reindex(hours_grid).values
        synthetic = profile_by_hour + day_residuals
        if target in NON_NEGATIVE_TARGETS:
            synthetic = np.clip(synthetic, 0.0, None)
        scenarios.append(synthetic)

    return hours_grid, profile_by_hour, np.array(scenarios), len(full_dates)


hours_grid, profile, scenarios, n_full_days = build_scenarios(merged[["timestamp", "wind_mw"]], "wind_mw", n_scenarios=30)
print(f"Resampling from {n_full_days} complete historical day(s) of wind_mw.")

fig, ax = plt.subplots(figsize=(9, 4))
for s in scenarios:
    ax.plot(hours_grid, s, color="tab:green", alpha=0.15, linewidth=1)
ax.plot(hours_grid, profile, color="black", linewidth=2, label="historical mean profile")
ax.set_xlabel("hour of day")
ax.set_ylabel("wind_mw")
ax.set_title(f"{scenarios.shape[0]} synthetic day-ahead wind scenarios (residual block bootstrap)")
ax.legend()
plt.tight_layout()
plt.show()


**What this is and isn't useful for**: the spread across these synthetic scenarios is a
reasonable stand-in for "how much could a day like this actually vary, based on what's
already been observed" -- useful as an input to stress-testing a battery dispatch policy
against more than just one point forecast, for instance. It is NOT a substitute for a
real numerical weather prediction ensemble, and with only a limited window of historical
days to resample from (see `n_full_days` above), the diversity of scenarios it can
produce is correspondingly limited -- more historical data would directly widen the pool
of distinct residual days available to splice in.



## Takeaways

- Wind (and daytime solar) are measurably more stochastic than load by both the lag-1
  autocorrelation and noise-ratio diagnostics in Section 1 -- this is a property of the
  underlying physical process, independent of any model.
- Section 2 extends `forecasting_theory.ipynb`'s calibration check (previously load-only)
  to all three targets in one comparable table, including a second calibration diagnostic
  (`pct_below_median`) that notebook didn't compute.
- Section 3 checks something the coverage number alone can't show: whether the
  uncertainty band's *width* actually varies sensibly with time of day (solar shrinking
  toward 0 at night being the clearest physically-expected pattern to look for).
- Section 4's scenario generator is intentionally simple and says so -- a real production
  system would want an actual weather ensemble, not a historical-residual bootstrap.

**A note on the numbers actually shown when you run this**: like `eda_grid_data.ipynb`
and `forecasting_theory.ipynb`, this notebook is fully live and regenerates every number
from whatever is currently in `data/processed/` and `outputs/` on your machine -- the
specific figures (though not the qualitative patterns) will differ with more or different
historical data than whatever was available when this was last run.
